# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [12]:
df = pd.read_csv(r"C:\Users\postm\OneDrive\Documents\Documents\Flatiron\aviationData\AviationData.csv", encoding="cp1252", low_memory=False)
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [13]:
df.columns = df.columns.str.lower().str.replace(r"[. ]+", "_", regex=True)
df.columns.tolist()
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["year"] = df["event_date"].dt.year

#Keep accidents from 1983 onward on professionally built planes
df = df[(df["year"] >= 1983)
    & (df["investigation_type"] == "Accident")
    & (df["amateur_built"] == "No")].copy()
df.shape

#--No fully identical rows exist. 938 repeated event_id values and 26 repeated accident_number values were found. All are multi-aircraft events where each row is a distinct aircraft (different registration, make, and model), so no rows were removed. 
#--Injury counts in these events appear to be event totals repeated on each aircraft's row, a limitation noted in the analysis. Filtered proffesionally built planes for 1983 and onward.
#--Rows missing make or model are removed in the Make and Model sections

(73286, 32)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [14]:
inj_cols = ["total_fatal_injuries", "total_serious_injuries", "total_minor_injuries", "total_uninjured"]

#Add the 4 injury columns for an estimate on total people aboard
df["total_aboard"] = df[inj_cols].sum(axis=1)

#Fatal and Serious injuries for each row where blanks are 0
fatal_serious = df["total_fatal_injuries"].fillna(0) + df["total_serious_injuries"].fillna(0)

#Turn rows with 0 people aboard into blanks since they have no data
aboard_safe = df["total_aboard"].where(df["total_aboard"] > 0)

#Rate = the amount of people aboard who had Serious or Fatal injuries
df["serious_fatal_rate"] = fatal_serious / aboard_safe

#Tests
print(df["serious_fatal_rate"].min(), df["serious_fatal_rate"].max()) 
print(df["serious_fatal_rate"].isna().sum())
df[["total_aboard", "serious_fatal_rate"]].head()

#--Estimated the people aboard by summing the four injury columns with blanks as 0. The serious_fatal_rate was calculated 
#--by adding Serious + Fatal injuries and dividing by people aboard. The rows with no people aboard were excluded


0.0 1.0
434


,total_aboard,serious_fatal_rate
3600,2.0,0.5
3601,4.0,0.0
3602,2.0,0.0
3603,1.0,0.0
3604,2.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [15]:
#Check values
print(df["aircraft_damage"].value_counts(dropna=False))

#Clean up "Unknown" and actually make it blank
df["aircraft_damage"] = df["aircraft_damage"].replace("Unknown", pd.NA)

#Create column for Destroyed where 1=Destroyed and 0=Substantial or Minor, blank stays blank
df["destroyed"] = df["aircraft_damage"].map({"Destroyed": 1, "Substantial": 0, "Minor": 0})
print(df["destroyed"].value_counts(dropna=False))

#--Cleaned up Unknown damaged values to blank. Created a new column called Destroyed for the different damage levels where destroyed=1 and substantial or minor=0
#--Missing damage data stays missing so it doesn't get counted as "Not Destroyed"

aircraft_damage
Substantial    55651
Destroyed      15462
NaN             1415
Minor            664
Unknown           94
Name: count, dtype: int64
destroyed
0.0    56315
1.0    15462
NaN     1509
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [16]:
#Remove spaces and uppercase spelling to match
df["make"] = df["make"].str.strip().str.upper()
df["make"].value_counts().head(40)

#Merge spelling variants of the same Makers
df["make"] = df["make"].replace({
    "ROBINSON HELICOPTER": "ROBINSON",
    "ROBINSON HELICOPTER COMPANY": "ROBINSON",
    "AIR TRACTOR INC": "AIR TRACTOR",
    "CIRRUS DESIGN CORP": "CIRRUS",
    "GRUMMAN AMERICAN": "GRUMMAN",
    "ERCOUPE (ENG & RESEARCH CORP.)": "ERCOUPE",
    "AIRBUS INDUSTRIE": "AIRBUS",
})

#Keeping makes with at least 50 records
counts = df["make"].value_counts()
df = df[df["make"].isin(counts[counts >= 50].index)]
df["make"].nunique(), len(df)

#--Cleaned up empty spaces and made all text uppercase. Merged spelling variants of the same maker and only kept makers with at least 50 record (77 makes and 66,866 rows remain).

(76, 66866)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [17]:
#Remove blank models and clean text
df = df.dropna(subset=["model"])
df["model"] = df["model"].str.strip().str.upper()

#Check amount of shared model names by more than 1 make
(df.groupby("model")["make"].nunique() > 1).sum()

#Make labels unique by plane type, make+model
df["plane_type"] = df["make"] + " " + df["model"]
df["plane_type"].value_counts().head(10)

#--Removed blank models. Model names were shared across make so plane_type was created to identify each aircraft by make+model. Close varients (Cessna 172 and 172N) are kept as seperate types.

plane_type
CESSNA 152         2215
CESSNA 172         1640
CESSNA 172N        1091
PIPER PA-28-140     862
CESSNA 172M         754
CESSNA 150          745
CESSNA 172P         661
CESSNA 182          611
CESSNA 180          594
PIPER PA-18         550
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [18]:
#Trim spaces, uppercase text and turn place holder words into blanks
text_cols = ["engine_type", "weather_condition", "purpose_of_flight", "broad_phase_of_flight"]

for col in text_cols:
    df[col] = df[col].str.strip().str.upper().replace(["UNK", "UNKNOWN"], pd.NA)
    print(df[col].value_counts(dropna=False), "\n")

#Make "LR" blank as its not a real engine type, just means "Long Range"
df["engine_type"] = df["engine_type"].replace("LR", pd.NA)

#Merge duplicate spelling and expand abbreviations for purpose_of_flight column
df["purpose_of_flight"] = df["purpose_of_flight"].replace({
    "AIR RACE SHOW": "AIR RACE/SHOW",
    "PUBS": "PUBLIC AIRCRAFT- STATE", 
    "PUBL": "PUBLIC AIRCRAFT- LOCAL",
})
df["purpose_of_flight"].value_counts().tail(10)
                                                          
df["number_of_engines"].value_counts(dropna=False)

#--Engine type "LR" set to blank. Merged AIR RACE SHOW with AIR RACE/SHOW as it's identiacal apart from punctuation. Expanded the abbreviations PUBS and PUBL to map to public aircraft state and local categories.

engine_type
RECIPROCATING    54401
NaN               4292
TURBO SHAFT       2896
TURBO PROP        2446
<NA>              1405
TURBO FAN         1098
TURBO JET          311
LR                   1
NONE                 1
Name: count, dtype: int64 

weather_condition
VMC     58541
IMC      5024
NaN      2488
<NA>      798
Name: count, dtype: int64 

purpose_of_flight
PERSONAL                     36638
INSTRUCTIONAL                 9278
<NA>                          4330
AERIAL APPLICATION            4128
NaN                           3592
BUSINESS                      3324
POSITIONING                   1368
OTHER WORK USE                1041
AERIAL OBSERVATION             704
PUBLIC AIRCRAFT                624
FERRY                          619
EXECUTIVE/CORPORATE            390
SKYDIVING                      174
FLIGHT TEST                    147
BANNER TOW                      97
EXTERNAL LOAD                   85
PUBLIC AIRCRAFT - FEDERAL       72
PUBLIC AIRCRAFT - STATE         56
PUB

number_of_engines
1.0    53984
2.0     8009
NaN     3756
0.0      725
4.0      200
3.0      176
8.0        1
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [19]:
#Drop columns that are more than 50% blank
cols_to_drop = df.columns[df.isna().mean() > 0.5]
print(list(cols_to_drop))
df = df.drop(columns=cols_to_drop)
df.shape

['latitude', 'longitude', 'aircraft_category', 'far_description', 'schedule', 'air_carrier']


(66851, 30)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [20]:
df.to_csv("cleaned_aviation_data.csv", index=False)